In [44]:
import torch

import sys

sys.path.append("../../training-inference/image_classifiers/")
from model_loader import create_model
from cifar10_loader import make_cifar10_dataloaders, get_cifar10_transforms
import numpy as np

base_best = "/media/santripta/Santripta 1/EuroVisSubmissionData/Training_random/checkpoints/resnet_cifar-r0.0/20251119_203732/best_e135.pt"
r10_best = "/media/santripta/Santripta 1/EuroVisSubmissionData/Training_random/checkpoints/resnet_cifar-r0.1/20251119_212927/best_e55.pt"

base = create_model("resnet18", num_classes=10, checkpoint=base_best, device="cuda:0")
r10 = create_model("resnet18", num_classes=10, checkpoint=r10_best, device="cuda:0")

train_tf, test_tf = get_cifar10_transforms()
train, test = make_cifar10_dataloaders(data_root="../../datasets/cifar/data", batch_size=128, shuffle=False, train_transform=train_tf, test_transform=test_tf)

Loaded checkpoint /media/santripta/Santripta 1/EuroVisSubmissionData/Training_random/checkpoints/resnet_cifar-r0.0/20251119_203732/best_e135.pt. Missing/Unexpected keys: <All keys matched successfully>
Loaded checkpoint /media/santripta/Santripta 1/EuroVisSubmissionData/Training_random/checkpoints/resnet_cifar-r0.1/20251119_212927/best_e55.pt. Missing/Unexpected keys: <All keys matched successfully>


In [45]:
with torch.no_grad():
    base.eval()
    r10.eval()

    X_base = []
    X_r10 = []
    y = []

    for x in test:
        labels = x['label'].cuda()
        images = x['image'].cuda()

        features_base = base(images)
        features_r10 = r10(images)

        X_base.append(features_base.cpu())
        X_r10.append(features_r10.cpu())
        y.append(labels.cpu())

    X_base = torch.cat(X_base, dim=0).numpy()
    X_r10 = torch.cat(X_r10, dim=0).numpy()
    y = torch.cat(y, dim=0).numpy()

In [ ]:
preds_base = np.argmax(X_base, axis=1)
preds_r10 = np.argmax(X_r10, axis=1)
correct_base = preds_base == y
correct_r10 = preds_r10 == y

((array([   0,    1,    2, ..., 9997, 9998, 9999], shape=(8440,)),),
 (array([   0,    1,    2, ..., 9996, 9997, 9999], shape=(7927,)),))

In [73]:
def softmax(x):
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e_x / e_x.sum(axis=1, keepdims=True)

confs_base = np.max(softmax(X_base), axis=1)
confs_r10 = np.max(softmax(X_r10), axis=1)

margins_base = np.sort(X_base, axis=1)[:, -1] - np.sort(X_base, axis=1)[:, -2]
margins_r10 = np.sort(X_r10, axis=1)[:, -1] - np.sort(X_r10, axis=1)[:, -2]

In [74]:
np.mean(margins_base), np.mean(margins_r10)

(np.float32(10.732537), np.float32(2.8323298))

In [70]:
np.mean(confs_base_correct), np.mean(confs_r10_correct)

(np.float32(0.9739148), np.float32(0.8615691))

In [65]:
confs_base_correct_correct = confs_base[correct_base & correct_r10]
confs_r10_correct_correct = confs_r10[correct_base & correct_r10]

np.mean(confs_base_correct_correct), np.mean(confs_r10_correct_correct)

(np.float32(0.9822095), np.float32(0.8724766))

In [67]:
confs_base_correct_incorrect = confs_base[correct_base & ~correct_r10]
confs_r10_correct_incorrect = confs_r10[correct_base & ~correct_r10]

np.mean(confs_base_correct_incorrect), np.mean(confs_r10_correct_incorrect)

(np.float32(0.9119917), np.float32(0.61986446))

In [69]:
confs_base[~correct_base].mean(), confs_r10[~correct_r10].mean()

(np.float32(0.8441911), np.float32(0.6504208))